In [27]:
import numpy as np
from tqdm import tqdm

from sklearn.metrics import classification_report

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
data = load_dataset('cornell-movie-review-data/rotten_tomatoes')

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map = "auto")

In [6]:
print(f"Training split : {len(data['train'])}")
print(f"Testing split : {len(data['test'])}")
print(f"validation split : {len(data['validation'])}")

Training split : 8530
Testing split : 1066
validation split : 1066


In [12]:
for i in map(int, np.random.randint(0, len(data['train']), 2)):
    print(f"Text {i}: {data['train']['text'][i]}")
    print(f"Label {i}: {data['train']['label'][i]}\n")

Text 1800: perhaps it's cliche to call the film 'refreshing , ' but it is . 'drumline' shows a level of young , black manhood that is funny , touching , smart and complicated .
Label 1800: 1

Text 6904: the period -- swinging london in the time of the mods and the rockers -- gets the once-over once again in gangster no . 1 , but falls apart long before the end .
Label 6904: 0



In [ ]:
prompt = "Is the following sentence positive or negative?"

data = data.map(lambda example : {"t5" : prompt + example['text']})

In [44]:
y_pred = []

for i, content in enumerate(tqdm(data['test']['t5'], total = len(data['test']))):
  try:
    inputs = tokenizer(content, return_tensors = "pt")
    inputs = {k: v.to(model.device) for k,v in inputs.items()}

    output = model.generate(**inputs, max_new_tokens = 5)

    prediction = tokenizer.decode(output[0], skip_special_tokens= True)

    y_pred.append(0 if prediction == "negative" else 1)

  except Exception as e:
    print(f"error at index {i} : {e}")
    break


100%|██████████| 1066/1066 [00:55<00:00, 19.30it/s]


In [53]:
def evaluate(y_true, y_pred):
  performance = classification_report(y_true, y_pred, target_names=['Negative Review', "Positive Review"], labels = [0,1])
  print(performance)

In [54]:
evaluate(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.83      0.84      0.83       533
Positive Review       0.84      0.83      0.83       533

       accuracy                           0.83      1066
      macro avg       0.83      0.83      0.83      1066
   weighted avg       0.83      0.83      0.83      1066

